In [1]:
import os
import h5py
import numpy as np
import pandas as pd
from scipy import signal
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import f1_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as SKLDA
import torch.nn as nn
from xgboost import XGBClassifier
#from skorch import NeuralNetClassifier
import torch.nn.functional as F
import torch
from sklearn.preprocessing import RobustScaler
#from skorch.callbacks import Callback
from sklearn.base import BaseEstimator, ClassifierMixin
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression

2025-10-21 06:46:58.444853: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761029218.721969      13 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761029218.800491      13 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
sample_rate = 2000
window_size_ms = 128#150
stride_ms = 50
window_size = int(window_size_ms / 1000 * sample_rate)
stride = int(stride_ms / 1000 * sample_rate)
#window_size = int(0.2 * sample_rate)  # 200 ms
#stride = int(0.025 * sample_rate)     # 25 ms
trial_duration = 5
samples_per_trial = trial_duration * sample_rate
num_channels = 16
num_grasps = 6
num_positions = 9

notch_freq = 50.0  # Hz, powerline noise (adjust to 60 if needed)
q = 30.0

con_plus = [2, 4, 5, 6, 8] 
con_x = [1, 3, 5, 7, 9]

In [ ]:
def load_data(participant, day, session, base_path='/kaggle/input/emg-data'):
    folder = f'participant_{participant}/day_{day}/session_{session}'
    path = os.path.join(base_path, folder)

    with h5py.File(os.path.join(path, 'emg_data.hdf5'), 'r') as f:
        emg = {key: np.array(f[key]) for key in f.keys()}

    labels_df = pd.read_csv(os.path.join(path, 'trials.csv'))

    #with open(os.path.join(path, 'recording_parameters.txt'), 'r') as f:
    #    metadata = f.read()
    
    return emg, labels_df

def load_and_prepare_day(participant, day, base_path, features='hudgin_5'):
    X_list = []
    yg_list = []
    yp_list = []
    for session in [1, 2]:
        emg, labels_df = load_data(participant, day, session, base_path)
        X, y_grasp, y_pos = prepare_dataset(emg, labels_df, features=features)
        X_list.append(X)
        yg_list.append(y_grasp)
        yp_list.append(y_pos)
    X_day = np.vstack(X_list)
    yg_day = np.concatenate(yg_list)
    yp_day = np.concatenate(yp_list)
    return X_day, yg_day, yp_day

def apply_bandpass(data, fs=sample_rate, low=20, high=450):
    sos = signal.butter(4, [low, high], btype='band', fs=fs, output='sos')
    filtered = signal.sosfiltfilt(sos, data, axis=1)
    return filtered 
    
def apply_notch_filter(data, fs=sample_rate, freq=notch_freq, q=q):
    b, a = signal.iirnotch(freq, q, fs)
    filtered = signal.filtfilt(b, a, data, axis=1)
    return filtered

def offset_correction(data):
    return data - np.mean(data, axis=1, keepdims=True)

def extract_windows(data):
    num_samples = data.shape[1] # 10000
    windows = []
    for start in range(0, num_samples - window_size + 1, stride):
        window = data[:, start:start + window_size]
        windows.append(window)
    return np.array(windows)

def extract_hudgins_9_features(window):
    if window.ndim != 2:
        raise ValueError("Window must be a 2D array (channels × samples)")
        
    mav = np.mean(np.abs(window), axis=1)
    zc = np.sum(np.diff(np.sign(window), axis=1) != 0, axis=1)
    ssc = np.sum(np.diff(np.sign(np.diff(window, axis=1)), axis=1) != 0, axis=1)
    wl = np.sum(np.abs(np.diff(window, axis=1)), axis=1)

    rms = np.sqrt(np.mean(window**2, axis=1))
    var = np.var(window, axis=1)
    std = np.std(window, axis=1)
    kurt = np.mean(((window - np.mean(window, axis=1, keepdims=True)) / std[:, None])**4, axis=1)
    skew = np.mean(((window - np.mean(window, axis=1, keepdims=True)) / std[:, None])**3, axis=1)

    features = np.concatenate([mav, zc, ssc, wl, rms, var, std, kurt, skew])
    return features

def extract_hudgins_5_features(window):
    if window.ndim != 2:
        raise ValueError("Window must be a 2D array (channels × samples)")
        
    mav = np.mean(np.abs(window), axis=1)
    zc = np.sum(np.diff(np.sign(window), axis=1) != 0, axis=1)
    ssc = np.sum(np.diff(np.sign(np.diff(window, axis=1)), axis=1) != 0, axis=1)
    wl = np.sum(np.abs(np.diff(window, axis=1)), axis=1)

    features = np.concatenate([mav, zc, ssc, wl])
    return features

def extract_sntdf_features(window):
    
    mav_nl = np.log(np.mean(np.abs(window), axis=1) + 1e-8)
    power0 = np.mean(window**0, axis=1)
    power2 = np.mean(window**2, axis=1)
    power4 = np.mean(window**4, axis=1)
    power6 = np.mean(window**6, axis=1)

    diff1 = np.mean(np.abs(np.diff(window, axis=1)), axis=1)
    diff2 = np.mean(np.abs(np.diff(np.diff(window, axis=1), axis=1)), axis=1)

    corr = np.corrcoef(window)
    corr_flat = corr[np.triu_indices(num_channels, k=1)]  # Upper triangle
    features = np.concatenate([mav_nl, power0, power2, power4, power6, diff1, diff2, corr_flat])
    return features

def extract_combine_features(window):
    features_sntdf = extract_sntdf_features(window)
    features_hudgins = extract_hudgins_9_features(window)
    
    combined_features = np.concatenate([features_sntdf, features_hudgins])    
    return combined_features

def process_trial(emg_trial, features='hudgin_5'):
    bandpassed = apply_bandpass(emg_trial)
    filtered = apply_notch_filter(bandpassed)
    #filtered = apply_notch_filter(emg_trial)
    corrected = offset_correction(filtered)
    windows = extract_windows(corrected)
    #for w in windows:
    #    print("w.shape: ", w.shape)
    #    break
    if features == 'hudgin_5':
        features = np.array([extract_hudgins_5_features(w) for w in windows])
    elif features == 'hudgin_9':
        features = np.array([extract_hudgins_9_features(w) for w in windows])
    elif features == 'sntdf':
        features = np.array([extract_sntdf_features(w) for w in windows])
    elif features == 'combine':
        features = np.array([extract_combine_features(w) for w in windows])
        
    return features

def prepare_dataset(emg, labels_df, features='hudgin_5'):
    features_list = []
    grasp_labels = []
    pos_labels = []

    for idx, row in labels_df.iterrows():
        trial_key = str(row['row_number'])
        #print(f"Row {idx}, trial_key: {trial_key}, in emg: {trial_key in emg}")
        if trial_key in emg:
            feat = process_trial(emg[trial_key], features=features)
            #print("len(feat): ", len(feat)) # 98
            num_windows = feat.shape[0]
            features_list.append(feat)
            grasp_labels.extend([row['grasp']] * num_windows)
            pos_labels.extend([row['target_position']] * num_windows)

    X = np.vstack(features_list)
    y_grasp = np.array(grasp_labels)
    y_pos = np.array(pos_labels)

    return X, y_grasp, y_pos

def split_data(X, y_grasp, y_pos, test_size=0.2):
    indices = np.arange(X.shape[0])
    X_train, X_test, yg_train, yg_test, yp_train, yp_test, idx_train, idx_test = train_test_split(
        X, y_grasp, y_pos, indices, test_size=test_size, random_state=200, stratify=y_grasp)
    
    return X_train, X_test, yg_train, yg_test, yp_train, yp_test

#def normalize_data(X_train, X_test):
#    scaler = StandardScaler()
#    X_train_norm = scaler.fit_transform(X_train)
#    X_test_norm = scaler.transform(X_test)
#    return X_train_norm, X_test_norm

def normalize_data(X_train, X_test):
    num_features = 4#9
    num_sensors = 4
    num_channels_per_sensor = 4
    X_train_norm = np.copy(X_train)
    X_test_norm = np.copy(X_test)
    for feat_idx in range(num_features):
        for sensor_idx in range(num_sensors):
            col_start = feat_idx * num_channels + sensor_idx * num_channels_per_sensor
            col_end = col_start + num_channels_per_sensor
            # Compute shared mean/std across the sensor's channels for this feature
            sensor_data = X_train[:, col_start:col_end]
            mean = np.mean(sensor_data)
            std = np.std(sensor_data)
            if std == 0:
                std = 1e-8  # Avoid division by zero
            X_train_norm[:, col_start:col_end] = (sensor_data - mean) / std
            # Apply to test
            sensor_data_test = X_test[:, col_start:col_end]
            X_test_norm[:, col_start:col_end] = (sensor_data_test - mean) / std
    return X_train_norm, X_test_norm
            
def evaluate_lda(X, y, n_splits=5):
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=200)
    accs = []
    for train_idx, val_idx in kf.split(X, y):
        clf = LinearDiscriminantAnalysis()
        clf.fit(X[train_idx], y[train_idx])
        pred = clf.predict(X[val_idx])
        accs.append(accuracy_score(y[val_idx], pred))
    return np.mean(accs)

def evaluate_svm(X, y, n_splits=5):
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=200)
    accs = []
    for train_idx, val_idx in kf.split(X, y):
        clf = SVC(kernel='rbf', C=1.0, gamma='scale')
        clf.fit(X[train_idx], y[train_idx])
        pred = clf.predict(X[val_idx])
        accs.append(accuracy_score(y[val_idx], pred))
    return np.mean(accs)

def standard_classification(X, y_grasp, y_pos):
    results = {}
    for pos in np.unique(y_pos):
        mask = (y_pos == pos)
        X_pos = X[mask]
        y_pos_grasp = y_grasp[mask]
        #print(f"Position {pos}: {len(np.unique(y_pos_grasp))} grasp classes, {len(y_pos_grasp)} samples")
        
        if len(X_pos) == 0:
            print(f"Warning: No data for position {pos}")
            continue
        X_train, X_test, y_train, y_test = train_test_split(X_pos, y_pos_grasp, test_size=0.2, random_state=200)
        #scaler = StandardScaler()
        #X_train = scaler.fit_transform(X_train)
        #X_test = scaler.transform(X_test)
        X_train, X_test = normalize_data(X_train, X_test)
        #cv_acc = evaluate_lda(X_train, y_train)
        cv_acc = evaluate_svm(X_train, y_train)

        clf = LinearDiscriminantAnalysis()
        #clf = SVC(kernel='rbf', C=1.0, gamma='scale')
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        test_acc = accuracy_score(y_test, y_pred)
        
        results[pos] = {"cv_acc": cv_acc, "test_acc": test_acc}
        
    #print("Standard Classification Accuracies (CV & Test):", results)
    return results

def naive_transfer(X, y_grasp, y_pos, config_positions):
    results = np.zeros((len(config_positions), len(config_positions)))
    for i, src in enumerate(config_positions):
        mask_src = (y_pos == src)
        X_src = X[mask_src]
        y_src = y_grasp[mask_src]

        if len(X_src) == 0:
            print(f"Warning: No data for source position {src}")
            continue

        #scaler = StandardScaler()
        #X_src_norm = scaler.fit_transform(X_src)
        X_src_norm, _ = normalize_data(X_src, X_src)

        clf = LinearDiscriminantAnalysis()
        #clf = SVC(kernel='rbf', C=1.0, gamma='scale')
        clf.fit(X_src_norm, y_src)

        for j, tgt in enumerate(config_positions):
            if i == j:
                #acc = evaluate_lda(X_src_norm, y_src)
                acc = evaluate_svm(X_src_norm, y_src)
            else:
                mask_tgt = (y_pos == tgt)
                X_tgt = X[mask_tgt]
                y_tgt = y_grasp[mask_tgt]
                if len(X_tgt) == 0:
                    print(f"Warning: No data for target position {tgt}")
                    continue
                    
                #X_tgt_norm = scaler.transform(X_tgt)
                _, X_tgt_norm = normalize_data(X_src, X_tgt)
                pred = clf.predict(X_tgt_norm)
                acc = accuracy_score(y_tgt, pred)
            results[i, j] = acc
            
    #print("Naive Transfer Matrix: ", results)
    return results

def position_classification(X, y_grasp, y_pos, config_positions):
    results = {}
    for grasp in range(1, num_grasps+1):
        mask = (y_grasp == grasp) & np.isin(y_pos, config_positions)
        X_g = X[mask]
        y_g_pos = y_pos[mask]
        if len(X_g) == 0:
            print(f"Warning: No data for grasp {grasp}")
            continue
        X_train, X_test, y_train, y_test = train_test_split(X_g, y_g_pos, test_size=0.2, random_state=200)
        X_train, X_test = normalize_data(X_train, X_test)
        #cv_acc = evaluate_lda(X_train, y_train)
        cv_acc = evaluate_svm(X_train, y_train)
        
        clf = LinearDiscriminantAnalysis()
        #clf = SVC(kernel='rbf', C=1.0, gamma='scale')
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        test_acc = accuracy_score(y_test, y_pred)

        results[grasp] = {"cv_acc": cv_acc, "test_acc": test_acc}

    #print("Position Classification Accuracies per Grasp (CV & Test):", results)
    return results

class MLP48(BaseEstimator, ClassifierMixin):
    def __init__(self, input_dim=None, n_classes=None, lr=1e-3, epochs=30, batch_size=32, verbose=0):
        self.input_dim = input_dim
        self.n_classes = n_classes
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.verbose = verbose
        self.model = None
        self.loss_history = []

    def _build_model(self):
        model = Sequential()
        model.add(Dense(48, activation='relu', input_dim=self.input_dim))
        model.add(BatchNormalization())
        model.add(Dropout(0.3))
        model.add(Dense(24, activation='relu'))
        model.add(BatchNormalization())
        model.add(Dropout(0.3))
        model.add(Dense(self.n_classes, activation='softmax'))

        model.compile(optimizer=Adam(learning_rate=self.lr),
                     loss='categorical_crossentropy',
                     metrics=['accuracy'])
        return model

    def fit(self, X, y):
        if self.input_dim is None:
            self.input_dim = X.shape[1]
        if self.n_classes is None:
            self.n_classes = len(np.unique(y))
        y_cat = to_categorical(y, num_classes=self.n_classes)
        self.model = self._build_model()

        history = self.model.fit(
            X, y_cat,
            epochs=self.epochs,
            batch_size=self.batch_size,
            verbose=self.verbose)
        self.loss_history = history.history['loss']  # 👈 Lưu loss
        return self

    def predict(self, X):
        preds = np.argmax(self.model.predict(X, verbose=0), axis=1)
        return preds

def meta_learner_ensemble(X_train, y_train, X_test, random_state=42, cv=4):

    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)

    svc = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=random_state)
    xgb = XGBClassifier(
        use_label_encoder=False, eval_metric='mlogloss', n_estimators=250,
        max_depth=5, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, random_state=random_state, verbosity=0
    )
    mlp48 = MLP48(
        input_dim=X_train.shape[1], n_classes=len(np.unique(y_train_enc)),
        epochs=30, verbose=0
    )

    base_models = [svc, xgb, mlp48]

    cv_split = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)
    oof_probas = np.zeros((X_train.shape[0], len(np.unique(y_train_enc)) * len(base_models)))
    test_probas = np.zeros((X_test.shape[0], len(np.unique(y_train_enc)) * len(base_models)))

    for i, model in enumerate(base_models):
        oof_fold_preds = np.zeros((X_train.shape[0], len(np.unique(y_train_enc))))
        test_fold_preds = np.zeros((cv, X_test.shape[0], len(np.unique(y_train_enc))))

        for j, (train_idx, val_idx) in enumerate(cv_split.split(X_train, y_train_enc)):
            X_tr, X_val = X_train[train_idx], X_train[val_idx]
            y_tr, y_val = y_train_enc[train_idx], y_train_enc[val_idx]

            if isinstance(model, MLP48):
                model_fold = MLP48(input_dim=X_train.shape[1],
                                   n_classes=len(np.unique(y_train_enc)),
                                   epochs=30, verbose=0)
                model_fold.fit(X_tr, y_tr)
                val_pred = model_fold.model.predict(X_val, verbose=0)
                test_pred = model_fold.model.predict(X_test, verbose=0)
            else:
                model_fold = model.__class__(**model.get_params())
                model_fold.fit(X_tr, y_tr)
                val_pred = model_fold.predict_proba(X_val)
                test_pred = model_fold.predict_proba(X_test)

            oof_fold_preds[val_idx] = val_pred
            test_fold_preds[j, :, :] = test_pred

        oof_probas[:, i * len(np.unique(y_train_enc)):(i + 1) * len(np.unique(y_train_enc))] = oof_fold_preds
        test_probas[:, i * len(np.unique(y_train_enc)):(i + 1) * len(np.unique(y_train_enc))] = test_fold_preds.mean(axis=0)

    scaler = MinMaxScaler()
    X_meta_train = scaler.fit_transform(oof_probas)
    X_meta_test  = scaler.transform(test_probas)

    meta_model = LogisticRegression(max_iter=500, random_state=random_state)
    meta_model.fit(X_meta_train, y_train_enc)

    #y_pred_train = meta_model.predict(X_meta_train)
    #y_pred_test  = meta_model.predict(X_meta_test)

    y_pred_train = le.inverse_transform(meta_model.predict(X_meta_train))
    y_pred_test  = le.inverse_transform(meta_model.predict(X_meta_test))

    return y_pred_train, y_pred_test

def hmc_classification(X, y_grasp, y_pos, test_size=0.2, n_splits=5, random_state=200): 

    X_train, X_test, yg_train, yg_test, yp_train, yp_test = train_test_split(X, y_grasp, y_pos, test_size=test_size, random_state=random_state, stratify=y_grasp)

    #scaler = StandardScaler()
    #X_train_norm = scaler.fit_transform(X_train)
    #X_test_norm = scaler.transform(X_test)
    X_train_norm, X_test_norm = normalize_data(X_train, X_test)

    scaler = StandardScaler()
    X_train_norm = scaler.fit_transform(X_train_norm)
    X_test_norm = scaler.transform(X_test_norm)

    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    pos_cv_scores = []
    multi_label_cv_scores = []
    soft_cv_scores = []

    for train_idx, val_idx in kf.split(X_train_norm, yp_train):
        X_cv_train, X_cv_val = X_train_norm[train_idx], X_train_norm[val_idx]
        yg_cv_train, yg_cv_val = yg_train[train_idx], yg_train[val_idx]
        yp_cv_train, yp_cv_val = yp_train[train_idx], yp_train[val_idx]

        #clf_pos = LinearDiscriminantAnalysis()
        #clf_pos = SVC(kernel='rbf', C=1.0, gamma='scale')
        #clf_pos.fit(X_cv_train, yp_cv_train)
        #pos_pred_cv = clf_pos.predict(X_cv_val)
        temp, pos_pred_cv = meta_learner_ensemble(X_cv_train, yp_cv_train, X_cv_val)
        pos_cv_acc = accuracy_score(yp_cv_val, pos_pred_cv)
        pos_cv_scores.append(pos_cv_acc)

        std = np.std(pos_pred_cv)
        pos_pred_norm_cv = (pos_pred_cv - np.mean(pos_pred_cv)) / (std + 1e-8) # Avoid div by zero
        X_aug_cv = np.hstack((X_cv_val, pos_pred_norm_cv.reshape(-1, 1)))
        X_aug_train_cv = np.hstack((X_cv_train,
            ((temp - np.mean(temp)) / 
             np.std(temp + 1e-8)).reshape(-1, 1)))

        grasp_clfs = {}
        unique_pos = np.unique(yp_cv_train)
        
        for pos in unique_pos:
            mask = (yp_cv_train == pos)
            X_pos_aug = X_aug_train_cv[mask]
            y_pos_grasp = yg_cv_train[mask]
            if len(np.unique(y_pos_grasp)) > 1:  # Ensure enough classes
                clf_g = LinearDiscriminantAnalysis()
                #clf_g = SVC(kernel='rbf', C=1.0, gamma='scale')
                clf_g.fit(X_pos_aug, y_pos_grasp)
                grasp_clfs[pos] = clf_g

        preds_grasp = []
        soft_preds = []
        for i in range(len(X_cv_val)):
            p_pred = pos_pred_cv[i]
            x_aug = X_aug_cv[i].reshape(1, -1)
            true_p = yp_cv_val[i]

            if p_pred in grasp_clfs:
                pred_g = grasp_clfs[p_pred].predict(x_aug)[0]
            else:
                pred_g = np.random.choice(np.unique(yg_cv_train))
                print("handle rare edge cases where a position might not have been trained due to data imbalance or filtering")
                
            preds_grasp.append(pred_g)

            if true_p in grasp_clfs:
                pred_g_soft = grasp_clfs[true_p].predict(x_aug)[0]
            else:
                pred_g_soft = np.random.choice(np.unique(yg_cv_train))
                print("handle rare edge cases where a position might not have been trained due to data imbalance or filtering")
            
            soft_preds.append(pred_g_soft)

        multi_label_acc = np.mean((pos_pred_cv == yp_cv_val) & (np.array(preds_grasp) == yg_cv_val))
        soft_acc = accuracy_score(yg_cv_val, soft_preds)
        #soft_acc = accuracy_score(yg_cv_val, preds_grasp)
        multi_label_cv_scores.append(multi_label_acc)
        soft_cv_scores.append(soft_acc)

    cv_pos_acc = np.mean(pos_cv_scores)
    cv_multi_label_acc = np.mean(multi_label_cv_scores)
    cv_soft_acc = np.mean(soft_cv_scores)

    # Final training on full training set
    #clf_pos = LinearDiscriminantAnalysis()
    #clf_pos = SVC(kernel='rbf', C=1.0, gamma='scale')
    #clf_pos.fit(X_train_norm, yp_train)
    #pos_pred_test = clf_pos.predict(X_test_norm)
    temp_new, pos_pred_test = meta_learner_ensemble(X_train_norm, yp_train, X_test_norm)
    pos_acc = accuracy_score(yp_test, pos_pred_test)
    
    pos_pred_norm = (pos_pred_test - np.mean(pos_pred_test)) / (np.std(pos_pred_test) + 1e-8)
    X_aug_test = np.hstack((X_test_norm, pos_pred_norm.reshape(-1, 1)))
    X_aug_train = np.hstack((X_train_norm,
        ((temp_new - np.mean(temp_new)) / 
         np.std(temp_new + 1e-8)).reshape(-1, 1)))

    grasp_clfs = {}
    unique_pos = np.unique(yp_train)
    for pos in unique_pos:
        mask = (yp_train == pos)
        X_pos_aug = X_aug_train[mask]
        y_pos_grasp = yg_train[mask]
        if len(np.unique(y_pos_grasp)) > 1:
            clf_g = LinearDiscriminantAnalysis()
            #clf_g = SVC(kernel='rbf', C=1.0, gamma='scale')
            clf_g.fit(X_pos_aug, y_pos_grasp)
            grasp_clfs[pos] = clf_g

    preds_grasp = []
    soft_preds = []
    for i in range(len(X_test_norm)):
        p_pred = pos_pred_test[i]
        x_aug = X_aug_test[i].reshape(1, -1)
        true_p = yp_test[i]
        
        # HMC prediction
        if p_pred in grasp_clfs:
            pred_g = grasp_clfs[p_pred].predict(x_aug)[0]
        else:
            pred_g = np.random.choice(np.unique(yg_train))
            print("handle rare edge cases where a position might not have been trained due to data imbalance or filtering")

        preds_grasp.append(pred_g)
        
        # Soft HMC
        if true_p in grasp_clfs:
            pred_g_soft = grasp_clfs[true_p].predict(x_aug)[0]
        else:
            pred_g_soft = np.random.choice(np.unique(yg_train))
            print("handle rare edge cases where a position might not have been trained due to data imbalance or filtering")

        soft_preds.append(pred_g_soft)

    multi_label_acc = np.mean((pos_pred_test == yp_test) & (np.array(preds_grasp) == yg_test))
    soft_acc = accuracy_score(yg_test, soft_preds)
    #soft_acc = accuracy_score(yg_test, preds_grasp)
    
    #print(f"CV Position Acc: {cv_pos_acc:.4f}, CV Multi-label Acc: {cv_multi_label_acc:.4f}, CV Soft Acc: {cv_soft_acc:.4f}")
    #print(f"Test Position Acc: {pos_acc:.4f}, Test Multi-label Acc: {multi_label_acc:.4f}, Test Soft Acc: {soft_acc:.4f}")
    
    return cv_pos_acc, cv_multi_label_acc, cv_soft_acc, pos_acc, multi_label_acc, soft_acc

def run_pipeline(participant=1, day=1, session=1, features='hudgin_5'):
    emg, labels_df = load_data(participant, day, session)
    X, y_grasp, y_pos = prepare_dataset(emg, labels_df, features=features)

    print("len(X): ", len(X)) # 14699

    config = con_plus if day == 1 else con_x

    standard_results = standard_classification(X, y_grasp, y_pos)
    transfer_results = naive_transfer(X, y_grasp, y_pos, config)
    pos_class_results = position_classification(X, y_grasp, y_pos, config)
    #hmc_results = hmc_classification(X, y_grasp, y_pos)

    #return {'standard': standard_results, 'transfer': transfer_results, 'pos_class': pos_class_results, 'hmc': hmc_results}

    return {'standard': standard_results, 'transfer': transfer_results, 'pos_class': pos_class_results}

def run_pipeline_hmc(participant=1, base_path='/kaggle/input/emg-data', features='hudgin_5'):

    X_day1, yg_day1, yp_day1 = load_and_prepare_day(participant, 1, base_path, features=features)
    X_day2, yg_day2, yp_day2 = load_and_prepare_day(participant, 2, base_path, features=features)

    mask_no5 = (yp_day2 != 5)
    X_merged = np.vstack((X_day1, X_day2[mask_no5]))
    yg_merged = np.concatenate((yg_day1, yg_day2[mask_no5]))
    yp_merged = np.concatenate((yp_day1, yp_day2[mask_no5]))

    print("Merged data shape:", X_merged.shape, "Unique positions:", np.unique(yp_merged))

    hmc_results = hmc_classification(X_merged, yg_merged, yp_merged)

    return {'hmc': hmc_results}
    
def load_participant_data(participant, base_path='/kaggle/input/emg-data', features='hudgin_5'):
    X_p = []
    y_grasp_p = []
    y_pos_p = []
    day_list = [1, 2]
    session_list = [1, 2]
    for day in day_list:
        for session in session_list:
            emg, labels_df = load_data(participant, day, session, base_path)
            
            if day == 2:
                labels_df = labels_df[labels_df["target_position"] != 5].reset_index(drop=True)
            
            X, y_grasp, y_pos = prepare_dataset(emg, labels_df, features=features)
            X_p.append(X)
            y_grasp_p.append(y_grasp)
            y_pos_p.append(y_pos)
    if X_p:
        X = np.vstack(X_p)
        y_grasp = np.concatenate(y_grasp_p) 
        y_pos = np.concatenate(y_pos_p)
    else:
        X, y_grasp, y_pos = np.array([]), np.array([]), np.array([])
    return X, y_grasp, y_pos
            
def run_full_pipeline(base_path='/kaggle/input/emg-data', features='hudgin_5'):
    participant_list = [1, 2, 3, 4, 5, 6, 7, 8]
    results_per_participant_grasp_classification = {}
    results_per_participant_position_classification = {}
    for participant in participant_list:
        print(f"Processing participant {participant}...")
        X, y_grasp, y_pos = load_participant_data(participant, base_path, features=features)
        if len(X) == 0:
            print(f"Warning: No data for participant {participant}")
            continue
        print(f"Participant {participant} data shape: X={X.shape}, y_grasp={y_grasp.shape}, y_pos={y_pos.shape}")
        results_per_participant_grasp_classification[participant] = standard_classification(X, y_grasp, y_pos)
        results_per_participant_position_classification[participant] = position_classification(X, y_grasp, y_pos, [1, 2, 3, 4, 5, 6, 7, 8, 9])
        
    positions_grasp = sorted(set(pos for res in results_per_participant_grasp_classification.values() for pos in res))
    grasp_position = sorted(set(grasp for res in results_per_participant_position_classification.values() for grasp in res))

    #print("position_grasp: ", positions_grasp) # [1, 2, 3, 4, 5, 6, 7, 8, 9]
    #print("grasp position: ", grasp_position) # [1, 2, 3, 4, 5, 6]
    
    average_results_grasp = {}
    average_results_position = {}
    
    for pos in positions_grasp:
        accs_grasp = [results_per_participant_grasp_classification[p].get(pos, np.nan)['test_acc'] for p in results_per_participant_grasp_classification if pos in results_per_participant_grasp_classification[p]]
        average_results_grasp[pos] = np.nanmean(accs_grasp)
    for grasp in grasp_position:
        accs_position = [results_per_participant_position_classification[p].get(grasp, np.nan)['test_acc'] for p in results_per_participant_position_classification if grasp in results_per_participant_position_classification[p]]
        average_results_position[grasp] = np.nanmean(accs_position)

    print("Average Standard Classification Accuracies over participants:", average_results_grasp)
    print("Average Position Classification Accuracies over participants:", average_results_position)

    return average_results_grasp, average_results_position

In [4]:
def summarize_results(all_results):
    # --- standard ---
    cv_standard = []
    test_standard = []
    for r in all_results['standard']:
        for fold in r.values():
            cv_standard.append(fold["cv_acc"])
            test_standard.append(fold["test_acc"])
    print("standard mean cv_acc:", np.mean(cv_standard))
    print("standard mean test_acc:", np.mean(test_standard))

    # --- transfer ---
    transfer_stack = np.stack(all_results['transfer'], axis=0)   # shape (N, m, n)
    transfer_overall_mean = np.mean(transfer_stack)
    transfer_matrix_mean = np.mean(transfer_stack, axis=0)

    print("transfer overall mean:", transfer_overall_mean)
    print("transfer mean matrix:\n", transfer_matrix_mean)

    # --- pos_class ---
    # gom kết quả theo từng pos
    pos_accs = {}  # {pos: {"cv": [...], "test": [...]}}

    for r in all_results['pos_class']:
        for pos, fold in r.items():
            if pos not in pos_accs:
                pos_accs[pos] = {"cv": [], "test": []}
            pos_accs[pos]["cv"].append(fold["cv_acc"])
            pos_accs[pos]["test"].append(fold["test_acc"])

    for pos, vals in pos_accs.items():
        print(f"pos_class {pos} mean cv_acc:", np.mean(vals["cv"]))
        print(f"pos_class {pos} mean test_acc:", np.mean(vals["test"]))

#all_results = {'standard': [], 'transfer': [], 'pos_class': []}

#for participant in [1,2,3,4,5,6,7,8]:
#    for day in [1,2]:
#        for session in [1,2]:
#            results = run_pipeline(participant=participant, day=day, session=session, features='hudgin_5')
#            all_results['standard'].append(results['standard'])
#            all_results['transfer'].append(results['transfer'])
#            all_results['pos_class'].append(results['pos_class'])

#summarize_results(all_results)

In [5]:
all_results_hmc = []

for participant in [1]:
    results_hmc = run_pipeline_hmc(participant=participant, features='combine')
    # results_hmc['hmc'] là tuple
    all_results_hmc.append(results_hmc['hmc'])

all_results_hmc = np.array(all_results_hmc)

cv_pos_accs        = all_results_hmc[:, 0]
cv_multi_label_accs = all_results_hmc[:, 1]
cv_soft_accs        = all_results_hmc[:, 2]
pos_accs           = all_results_hmc[:, 3]
multi_label_accs   = all_results_hmc[:, 4]
soft_accs          = all_results_hmc[:, 5]

print("run_pipeline_hmc mean CV Position Acc:", np.mean(cv_pos_accs))
print("run_pipeline_hmc mean CV Multi-label Acc:", np.mean(cv_multi_label_accs))
print("run_pipeline_hmc mean CV Soft Acc:", np.mean(cv_soft_accs))
print("run_pipeline_hmc mean Test Position Acc:", np.mean(pos_accs))
print("run_pipeline_hmc mean Test Multi-label Acc:", np.mean(multi_label_accs))
print("run_pipeline_hmc mean Test Soft Acc:", np.mean(soft_accs))

Merged data shape: (52924, 376) Unique positions: [1 2 3 4 5 6 7 8 9]


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2025-10-21 07:37:26.274150: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When u

run_pipeline_hmc mean CV Position Acc: 0.9591157595977247
run_pipeline_hmc mean CV Multi-label Acc: 0.9513923683483947
run_pipeline_hmc mean CV Soft Acc: 0.988899106977831
run_pipeline_hmc mean Test Position Acc: 0.9609825224374114
run_pipeline_hmc mean Test Multi-label Acc: 0.9544638639584317
run_pipeline_hmc mean Test Soft Acc: 0.9902692489371753


In [6]:
#run_full_pipeline()